In [ ]:
import numpy as np
import json
# with open(r'C:\Users\juani\Documents\Github\Abaqus_WELL_\wellClosure_axi.json') as f:
with open(r'C:\Users\leticia\Documents\GitHub\Abaqus_WELL\wellbore_closure_planestrain.json') as f:
    data = json.load(f)

def process_lithology(data, global_depth):

    # filtered_rocks = {}
    # filtered_layers = []
    l_depth = data["AnalysisData"]["Depth"]
    lithology = data["Lithology"]
    json_rocks = data["Rocks"]

    for layer in lithology:
        l_top = layer["Top"] 
        l_bottom = layer["Bottom"]
        
        if l_bottom <= l_depth and l_top <= l_depth:
            print(l_top, l_bottom)
        else:
            continue
            
        # Formata o nome da camada
        # Prepara o material da rocha
        rock_name = layer["Rock"]
        rock_mat = json_rocks[rock_name].copy()
        mat_name = "LAYER_%s" % rock_name # Ex: LAYER_SANDSTONE
        print(rock_mat.keys())
        rock_mat["Name"] = mat_name  
        # filtered_rocks[mat_name] = rock_mat
       
        print(f"Processed layer: {mat_name} with top at {l_depth} inches") 
        print(f"Original layer: {rock_name} at {l_depth} inches")
        # filtered_layers.append(new_layer)
    
    return global_depth, l_depth

global_depth = data["AnalysisData"]["Depth"]
print(f"Global depth for analysis: {global_depth} inches")
global_depth, l_depth = process_lithology(data, global_depth)

print("Finished processing lithology.")

Global depth for analysis: 2500.0 meters
2000.0 2500.0
dict_keys(['MohrCoulombParameters', 'Law', 'Elasticity', 'ElasticParameters', 'ThermalParameters'])
Processed layer: LAYER_SHALE with top at 2500.0 meters
Original layer: SHALE at 2500.0 meters
Finished processing lithology.


In [52]:
import os
import json
import sys

path_project = r'C:\Users\leticia\Documents\GitHub\Abaqus_WELL'

# Reading the json file and filling the input data for the analysis ####################
with open(r'C:\Users\leticia\Documents\GitHub\Abaqus_WELL\wellbore_closure_planestrain.json') as f:
    data = json.load(f)
    # print(f"Data keys: {data.keys()}")
# Data keys: dict_keys(['AnalysisData', 'ThermalGradient',
#           'Tubulars', 'Lithology', 'InSituStresses', 'Rocks', 'Cements',
#           'SteelGrades', 'Phases', 'Events', 'Fluids'])

# Variables read from json (geometry) #####################################

# name_phase = '3dda7930-6dbf-4d05-87f2-d2809a3e9fc6'
if "Phases" in data["AnalysisData"]:
    name_phase = data["AnalysisData"]["Phases"]
    print(name_phase)
phase_data = data["Phases"][name_phase]
print(phase_data)
if phase_data:
    name_tubular = phase_data["Casing"][0]["Tubular"]
    print(f"Tubular used in phase '{name_phase}': {name_tubular}")
else:
    print(f"Phase '{name_phase}' not found in data['Phases']")

diameter_wellbore = phase_data["HoleDiameter"]
print(diameter_wellbore)

812c492a-c945-4184-be51-841c5fb86b15
{'Casing': [{'Tubular': 'VAM_20_#169_K55', 'Top': 2000.0, 'Bottom': 3200.0}], 'CementSheath': [{'Cement': 'CLASS_G', 'Top': 2500.0, 'Bottom': 3200.0}], 'HoleDiameter': 26.0, 'Standoff': 100.0, 'RateOfPenetration': 10.0, 'TripSpeed': 400.0, 'Fluid': 'drilling 20 in', 'Name': 'surface'}
Tubular used in phase '812c492a-c945-4184-be51-841c5fb86b15': VAM_20_#169_K55
26.0


In [7]:
# from abaqusConstants import *
# from abaqus import *
import json
import sys

# path_project = r'C:\Users\juani\Documents\Github\Abaqus_WELL_'
path_project = r'C:\Users\leticia\Documents\GitHub\Abaqus_WELL'

if path_project not in sys.path:
    sys.path.append(path_project)

# from GEOMETRY_PS.geometry_PS import *

# mdb.models.changeKey(fromName='Model-1', toName='MyFirstModel')
# if 'MyFirstModel' not in mdb.models:
#     mdb.Model(name='MyFirstModel')

# Reading the json file and filling the input data for the analysis ####################
with open(r'C:\Users\leticia\Documents\GitHub\Abaqus_WELL\wellbore_closure_planestrain.json') as f:
    data = json.load(f)
    print(f"Data keys: {data.keys()}")
# Data keys: dict_keys(['AnalysisData', 'ThermalGradient',
#           'Tubulars', 'Lithology', 'InSituStresses', 'Rocks', 'Cements',
#           'SteelGrades', 'Phases', 'Events', 'Fluids'])

# Variables read from json (geometry) #####################################

if "Phases" not in data["AnalysisData"]:
    print("Chave 'Phases' não encontrada")
    print(data["AnalysisData"].keys())
name_phase = data["AnalysisData"]["Phases"]
print(name_phase)
phase_data = data["Phases"][name_phase]
if phase_data:
    name_tubular = phase_data["Casing"][0]["Tubular"]
else:
    print(f"Phase '{name_phase}' not found in data['Phases']")

############ Rock dimensions ############################
diameter_wellbore = phase_data["HoleDiameter"]
outer_radius_wellbore = diameter_wellbore / 2
outer_radius_wellbore = outer_radius_wellbore * 0.0254  # Convert from inches to meters
thickness_wellbore = outer_radius_wellbore * 0.98 # Variavel da espessura da rocha
inner_radius_wellbore = outer_radius_wellbore - thickness_wellbore

print(
f"O Radius of wellbore: {outer_radius_wellbore} inches",
f"I Radius of wellbore: {inner_radius_wellbore} inches",
f"Thickness of wellbore: {thickness_wellbore} inches"
)
########### Casing / Pipe dimensions ####################
outer_diameter_pipe = data["Tubulars"][name_tubular]['OD']
outer_radius_pipe = outer_diameter_pipe / 2
outer_radius_pipe = outer_radius_pipe * 0.0254  # Convert from inches to meters
thickness_pipe = data["Tubulars"][name_tubular]['Thickness']
thickness_pipe = thickness_pipe * 0.0254  # Convert from inches to meters
inner_radius_pipe = outer_radius_pipe - thickness_pipe

print(
f"O Radius of pipe: {outer_radius_pipe} inches",
f"I Radius of pipe: {inner_radius_pipe} inches",
f"Thickness of pipe: {thickness_pipe} inches"
)
########## Annulus dimensions ###########################
outer_radius_annular = inner_radius_wellbore
inner_radius_annular = outer_radius_pipe
thickness_annular = outer_radius_annular - inner_radius_annular
thickness_annular = thickness_annular * 0.0254  # Convert from inches to meters

print(
f"O Radius of annular: {outer_radius_annular} inches",
f"I Radius of annular: {inner_radius_annular} inches",
f"Thickness of annular: {thickness_annular} inches"
)

l_depth = data["AnalysisData"]["Depth"]
print(f"The bottom of the wellbore is at: {-l_depth} meters")

Data keys: dict_keys(['AnalysisData', 'ThermalGradient', 'Tubulars', 'Lithology', 'InSituStresses', 'Rocks', 'Cements', 'SteelGrades', 'Phases', 'Events', 'Fluids'])
812c492a-c945-4184-be51-841c5fb86b15
O Radius of wellbore: 0.3302 inches I Radius of wellbore: 0.006603999999999999 inches Thickness of wellbore: 0.323596 inches
O Radius of pipe: 0.254 inches I Radius of pipe: 0.2333752 inches Thickness of pipe: 0.020624800000000002 inches
O Radius of annular: 0.006603999999999999 inches I Radius of annular: 0.254 inches Thickness of annular: -0.0062838584 inches
The bottom of the wellbore is at: -2500.0 meters
